# Monreader pt. 1

In [2]:
import csv
import easyocr
import numpy as np
import os
import random
import shutil
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB3, ResNet50
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ModuleNotFoundError: No module named 'tensorflow'

## Data Analysis

Running analytics on the training and testing images being used was considered important so that we understand what we are working with. I initially set-up a query to get the counts and find out how images were in each of the flip and non-flip folders for the training and testing sets. The final counts suggested that there a little over several thousand images that were a part of the training set. This is important because having a large number in the training set can help when it comes time to test.

In [3]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

def count_images(directory):
    return len([name for name in os.listdir(directory) if os.path.isfile(os.path.join(directory, name))])

flip_dir = 'C:\\Users\\crozi\\images\\training\\flip'
nonflip_dir = 'C:\\Users\\crozi\\images\\training\\notflip'

num_flip_images = count_images(flip_dir)
num_nonflip_images = count_images(nonflip_dir)

print(f"Number of flip images in the training set: {num_flip_images}")
print(f"Number of non-flip images in the training set: {num_nonflip_images}")

#Counts of the images being used in the flip and non-flip folders for test and train
flip_dir_test = 'C:\\Users\\crozi\\images\\testing\\flip'
nonflip_dir_test ='C:\\Users\\crozi\\images\\testing\\notflip'

num_flip_images_test = count_images(flip_dir_test)
num_nonflip_images_test = count_images(nonflip_dir_test)

print(f"Number of flip images in the testing set: {num_flip_images_test}")
print(f"Number of non-flip images in the testing set: {num_nonflip_images_test}")


Number of flip images in the training set: 1162
Number of non-flip images in the training set: 1230
Number of flip images in the testing set: 290
Number of non-flip images in the testing set: 1


In [4]:
def show_random_images(folder_path, label, n=5):
    images = [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if len(images) == 0:
        print(f"No images found in {folder_path}")
        return

    images = random.sample(images, min(n, len(images)))

    plt.figure(figsize=(15, 4))
    for i, img_path in enumerate(images):
        img = Image.open(img_path)
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(label)
    plt.show()

In [1]:
#Simple test to visualize several of the random images within the train and test groups for flip and non-flip
show_random_images(flip_dir, "TRAIN: flip", n=5)
show_random_images(nonflip_dir, "TRAIN: notflip", n=5)

show_random_images(flip_dir_test, "TEST: flip", n=5)
show_random_images(nonflip_dir_test, "TEST: notflip", n=5)

NameError: name 'show_random_images' is not defined

The above shows random images pulled from the train and test images and used to help identify which images are appropriately labeled as 'flip' or 'nonflip'. The random images pulled show that the test images used are coming back with a degree of accuracy and that we are correctly identifying when pages are being turned or not turned.

In [6]:
source_dir = 'C:\\Users\\crozi\\images\\testing'
test_dir = 'C:\\Users\\crozi\\images\\testing_validation'

categories = ['flip', 'notflip']

if not os.path.exists(test_dir):
    os.mkdir(test_dir)

for category in categories:
    category_dir = os.path.join(test_dir, category)
    if not os.path.exists(category_dir):
        os.mkdir(category_dir)

split_ratio = 0.5

for category in categories:
    category_path = os.path.join(source_dir, category)
    images = os.listdir(category_path)
    random.shuffle(images)

split_index = int(len(images) * split_ratio)

images_to_move = images[:split_index]

for image in images_to_move:
    src_path = os.path.join(category_path, image)
    dest_path = os.path.join(test_dir, category, image)
    shutil.move(src_path, dest_path)

In [7]:
train_data_dir = 'C:\\Users\\crozi\\images\\images\\training'
test_data_dir = 'C:\\Users\\crozi\\images\\images\\testing'
test_set_data_dir = 'C:\\Users\\crozi\\images\\images\\testing_validation'

train_datagen = ImageDataGenerator(
     preprocessing_function=resnet50_preprocess,
     rotation_range=40,
     width_shift_range=0.2,
     height_shift_range=0.2,
     shear_range=0.2,
     zoom_range=0.2,
     horizontal_flip=True,
     fill_mode='nearest'
 )

test_datagen = ImageDataGenerator(
    preprocessing_function=resnet50_preprocess
)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

validation_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    test_set_data_dir,
    target_size=(224, 224),
    batch_size=1,
    class_mode='binary',
    shuffle=False)

Found 2392 images belonging to 2 classes.
Found 291 images belonging to 2 classes.
Found 306 images belonging to 2 classes.


The above charts show the training and validation for the images used. It contains both the trend lines for accuracy and loss and shows where they are heading as more and more testing runs are used. While they start with initially very high losses and somewhat inaccurate, the more times they are tested, the more that there tends to be an increase in the accuracy of the models and lower loss. There are occasional spikes, but following trends, the more we can expect that more testing will improve the overall quality though it may come at the cost of time.

In [18]:
class ChrisNet(tf.keras.Model):
    def __init__(self, dropout=0.35, l2=1e-4):
        super().__init__()
        reg = tf.keras.regularizers.l2(l2)

        self.block1 = tf.keras.Sequential([
            layers.Conv2D(32, 3, padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D()
        ])

        self.block2 = tf.keras.Sequential([
            layers.Conv2D(64, 3, padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D()
        ])

        self.block3 = tf.keras.Sequential([
            layers.Conv2D(128, 3, padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D()
        ])

        self.block4 = tf.keras.Sequential([
            layers.Conv2D(256, 3, padding="same", kernel_regularizer=reg),
            layers.BatchNormalization(),
            layers.ReLU()
        ])

        self.head = tf.keras.Sequential([
            layers.GlobalAveragePooling2D(),
            layers.Dense(128, activation="relu", kernel_regularizer=reg),
            layers.Dropout(dropout),
            layers.Dense(1, activation="sigmoid")
        ])

    def call(self, x, training=False):
        x = self.block1(x, training=training)
        x = self.block2(x, training=training)
        x = self.block3(x, training=training)
        x = self.block4(x, training=training)
        return self.head(x, training=training)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

eval_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

validation_generator = eval_datagen.flow_from_directory(
    test_data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_generator = eval_datagen.flow_from_directory(
    test_set_data_dir,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode="binary",
    shuffle=False
)

print("Class indices:", train_generator.class_indices)

model = ChrisNet(dropout=0.35, l2=1e-4)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint("chrisnet_best.keras", monitor="val_auc", mode="max", save_best_only=True),
]

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=50,
    callbacks=callbacks
)

test_metrics = model.evaluate(test_generator, verbose=1)
print("Holdout test metrics:", dict(zip(model.metrics_names, test_metrics)))

Found 2392 images belonging to 2 classes.
Found 306 images belonging to 2 classes.
Found 306 images belonging to 2 classes.
Class indices: {'flip': 0, 'notflip': 1}
Epoch 1/50
75/75 ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿

In [19]:
FOLDER_PATH = r"C:\Users\crozi\images\images\Text"
EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp', '.heic')

reader = easyocr.Reader(['en'], gpu=False)

folder = Path(FOLDER_PATH)
image_paths = [p for p in folder.rglob("*") if p.suffix.lower() in EXTS and p.stat().st_size > 5000]

print(f"Found {len(image_paths)} usable images.")

if not image_paths:
    print("No valid images found.")
else:
    img_path = random.choice(image_paths)
    print(f"\nSelected image: {img_path.name}")

    im = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    img = np.array(im)

    detections = reader.readtext(img, detail=2, paragraph=False)

    print("\n--- OCR Result ---")
    if not detections:
        print("(no text detected)")
    else:
        for bbox, text, conf in detections:
            print(f"{conf:.2f} | {text}")

Using CPU. Note: This module is much faster with a GPU.


Found 1 usable images.

Selected image: aiwplain1.jpg


C:\Users\crozi\PyCharmMiscProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- OCR Result ---
0.86 | CHAPTER I
0.71 | Down the Rabbit-Hole
0.83 | Alice was beginning to
0.94 | very tired of sitting by her sister on the
0.62 | bank, and of having nothing to do: once or twice she had peeped
0.76 | into the book her sister was reading, but it had no pictures or
0.66 | conversations in it,
0.78 | and what is the use of a book;' thought Alice
0.93 | 'without pictures or conversation?'
0.80 | So she was considering in her own mind (as well as she could, for
0.99 | the hot
0.82 | made her feel very sleepy and stupid), whether the
0.82 | pleasure of making a daisy-chain would be worth the trouble of
0.83 | getting up and picking the daisies, when suddenly a White Rabbit
0.95 | with pink eyes ran close by her
0.93 | There was nothing so VERY remarkable in that; nor did Alice think
0.59 | it so VERY much out of the way to hear the Rabbit say to itself; 'Oh
0.85 | dear! Oh dear! I shall be latel' (when she thought it over
0.93 | afterwards, it occurred to her that she 